# ASTGCN 云端消融实验训练与评估

本 notebook 用于在 Kaggle、Colab 或其他云端环境中批量训练 ASTGCN 消融实验，并生成指标表、柱状图和预测曲线。默认使用 `quick` 模式做流程验证，正式实验时将 `RUN_MODE` 改为 `full`。

In [ ]:
!git clone https://github.com/Tuzfucius/ASTGCN-learning

# 如云端环境未安装依赖，先取消下一行注释执行。
# %pip install -q numpy pandas PyYAML torch matplotlib tqdm scikit-learn

from __future__ import annotations

import copy
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

plt.rcParams["figure.dpi"] = 120
torch.backends.cudnn.benchmark = True

## 1. 项目路径与运行参数

如果云端工作目录不是项目根目录，请修改 `PROJECT_ROOT`。数据路径默认读取配置文件，也可以通过 `DATA_ROOT` 环境变量覆盖。

In [ ]:
PROJECT_ROOT = Path.cwd() / "ASTGCN-learning"
if PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "pems04.yaml"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "ablation_cloud"
RUN_MODE = "full"  # quick 或 full

QUICK_EPOCHS = 1
QUICK_MAX_BATCHES = 2
FULL_EPOCHS = 10  # full 模式下每个实验最多训练 10 epoch
FULL_MAX_BATCHES = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = PROJECT_ROOT / "data/raw/PEMS04"

sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("DEVICE:", DEVICE)

## 2. 导入项目模块

In [ ]:
from astgcn.data.dataloader import build_dataloaders
from astgcn.data.graph import build_graph_data
from astgcn.engine.evaluator import Evaluator
from astgcn.engine.predictor import Predictor
from astgcn.engine.trainer import Trainer
from astgcn.logger import get_logger
from astgcn.utils import ensure_dir, load_config, set_random_seed
import train as train_entry

base_config = load_config(CONFIG_PATH)
base_config["train"]["device"] = DEVICE
base_config["train"]["num_workers"] = min(int(base_config["train"].get("num_workers", 0)), 2)

if DATA_ROOT:
    data_root = Path(DATA_ROOT)
    base_config["dataset"]["data_path"] = str(data_root / "pems04.npz")
    base_config["dataset"]["distance_path"] = str(data_root / "distance.csv")

base_config

## 3. 消融实验列表

可在 `ABLATION_EXPERIMENTS` 中增删实验。每个实验只需要写相对完整模型的差异项。

In [ ]:
ABLATION_EXPERIMENTS = [
    {
        "name": "full_astgcn",
        "ablation": {},
    },
    {
        "name": "recent_only",
        "ablation": {"use_daily": False, "use_weekly": False},
    },
    {
        "name": "no_temporal_attention",
        "ablation": {"use_temporal_attention": False},
    },
    {
        "name": "no_spatial_attention",
        "ablation": {"use_spatial_attention": False},
    },
    {
        "name": "identity_graph",
        "ablation": {"graph_mode": "identity"},
    },
]

pd.DataFrame([{"name": item["name"], **item["ablation"]} for item in ABLATION_EXPERIMENTS]).fillna("")

## 4. 训练、评估与保存函数

In [ ]:
def deep_update(target: dict, updates: dict) -> dict:
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            deep_update(target[key], value)
        else:
            target[key] = value
    return target


def prepare_experiment_config(experiment: dict) -> dict:
    config = copy.deepcopy(base_config)
    config.setdefault("ablation", {})
    deep_update(config["ablation"], experiment.get("ablation", {}))
    exp_dir = OUTPUT_ROOT / experiment["name"]
    config["log"]["save_dir"] = str(exp_dir)
    config["log"]["checkpoint_dir"] = str(exp_dir / "checkpoints")
    config["log"]["log_dir"] = str(exp_dir / "logs")
    config["log"]["prediction_dir"] = str(exp_dir / "predictions")
    return config


def build_loaders_and_graph(config: dict):
    dataset_cfg = config["dataset"]
    window_cfg = config["time_window"]
    train_cfg = config["train"]
    graph_cfg = config["graph"]
    train_loader, val_loader, test_loader, scaler = build_dataloaders(
        data_path=dataset_cfg["data_path"],
        num_recent=window_cfg["recent_len"],
        num_days=window_cfg["daily_days"],
        num_weeks=window_cfg["weekly_weeks"],
        pred_len=window_cfg["pred_len"],
        batch_size=train_cfg["batch_size"],
        points_per_day=dataset_cfg["points_per_day"],
        target_dim=dataset_cfg["target_dim"],
        train_ratio=config["split"]["train_ratio"],
        val_ratio=config["split"]["val_ratio"],
        num_workers=train_cfg.get("num_workers", 0),
    )
    graph_data = build_graph_data(
        file_path=dataset_cfg["distance_path"],
        k_order=graph_cfg["cheb_order"],
        num_nodes=dataset_cfg["num_nodes"],
        directed=graph_cfg.get("directed", False),
        weighted=graph_cfg.get("weighted", False),
    )
    return train_loader, val_loader, test_loader, scaler, graph_data


def run_experiment(experiment: dict) -> dict:
    config = prepare_experiment_config(experiment)
    set_random_seed(int(config["train"].get("seed", 42)))
    exp_dir = Path(config["log"]["save_dir"])
    ensure_dir(exp_dir)
    ensure_dir(config["log"]["checkpoint_dir"])
    ensure_dir(config["log"]["log_dir"])
    ensure_dir(config["log"]["prediction_dir"])
    with open(exp_dir / "config.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)

    logger = get_logger(f"astgcn.ablation.{experiment['name']}", Path(config["log"]["log_dir"]) / "train.log")
    train_loader, val_loader, test_loader, scaler, graph_data = build_loaders_and_graph(config)
    model = train_entry.build_model(config, graph_data["chebyshev_polynomials"], torch.device(DEVICE))
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(config["train"]["learning_rate"]),
        weight_decay=float(config["train"].get("weight_decay", 0.0)),
    )
    trainer = Trainer(
        model=model,
        optimizer=optimizer,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        loss_name=config["train"].get("loss", "mae"),
        logger=logger,
        checkpoint_dir=config["log"]["checkpoint_dir"],
        config=config,
        scaler=scaler,
    )
    epochs = QUICK_EPOCHS if RUN_MODE == "quick" else (FULL_EPOCHS or int(config["train"]["epochs"]))
    max_batches = QUICK_MAX_BATCHES if RUN_MODE == "quick" else FULL_MAX_BATCHES
    history = trainer.fit(
        epochs=epochs,
        patience=config["train"].get("early_stop_patience"),
        max_batches=max_batches,
    )
    evaluator = Evaluator(
        model=model,
        data_loader=test_loader,
        scaler=scaler,
        device=DEVICE,
        target_dim=config["dataset"]["target_dim"],
    )
    test_metrics = evaluator.evaluate(max_batches=max_batches)
    pred_path = Path(config["log"]["prediction_dir"]) / "test_predictions.npz"
    Predictor(model=model, data_loader=test_loader, device=DEVICE).predict(
        pred_path,
        save_components=True,
        max_batches=max_batches,
    )
    result = {
        "experiment": experiment["name"],
        "epochs": epochs,
        "best_epoch": history.get("best_epoch"),
        "best_val_mae": history.get("best_metric"),
        **{f"test_{key}": value for key, value in test_metrics.items()},
        "prediction_path": str(pred_path),
    }
    with open(exp_dir / "result.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    return result

## 5. 批量运行消融实验

In [ ]:
results = []
for experiment in ABLATION_EXPERIMENTS:
    print(f"\n===== {experiment['name']} =====")
    result = run_experiment(experiment)
    results.append(result)
    display(pd.DataFrame(results))

results_df = pd.DataFrame(results)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
results_csv = OUTPUT_ROOT / "ablation_results.csv"
results_df.to_csv(results_csv, index=False)
results_df

## 6. 指标可视化

In [ ]:
metric_cols = [col for col in ["test_mae", "test_rmse", "test_mape"] if col in results_df.columns]
fig, axes = plt.subplots(1, len(metric_cols), figsize=(5 * len(metric_cols), 4), squeeze=False)
for ax, metric in zip(axes[0], metric_cols):
    ordered = results_df.sort_values(metric)
    ax.barh(ordered["experiment"], ordered[metric], color="#4C78A8")
    ax.set_title(metric)
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 7. 单节点预测曲线对比

In [ ]:
def plot_prediction_curves(results_df: pd.DataFrame, sample_idx: int = 0, node_idx: int = 0):
    plt.figure(figsize=(10, 5))
    target_plotted = False
    for _, row in results_df.iterrows():
        data = np.load(row["prediction_path"])
        pred = data["prediction"]
        target = data["target"]
        if sample_idx >= pred.shape[0] or node_idx >= pred.shape[1]:
            continue
        horizon = np.arange(pred.shape[-1])
        plt.plot(horizon, pred[sample_idx, node_idx], marker="o", linewidth=1.2, label=row["experiment"])
        if not target_plotted:
            plt.plot(horizon, target[sample_idx, node_idx], color="black", marker="x", linewidth=2.0, label="target")
            target_plotted = True
    plt.xlabel("预测步")
    plt.ylabel("交通流量")
    plt.title(f"样本 {sample_idx} / 节点 {node_idx} 的预测曲线")
    plt.grid(alpha=0.25)
    plt.legend(ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()

plot_prediction_curves(results_df, sample_idx=0, node_idx=0)

## 8. 相对完整模型的变化率

In [ ]:
baseline_name = "full_astgcn"
baseline = results_df[results_df["experiment"] == baseline_name].iloc[0]
summary = results_df.copy()
for metric in metric_cols:
    summary[f"{metric}_change_pct"] = (summary[metric] - baseline[metric]) / baseline[metric] * 100.0
summary.sort_values("test_mae" if "test_mae" in summary.columns else metric_cols[0])